# Swaption Vol Surface — SABR Model

This notebook covers the SABR stochastic volatility model and swaption pricing using `bond.vol`.

**Topics**
- SABR model: parameters and intuition
- Fitting SABR to market vol smiles
- Building a full expiry × strike vol surface
- Black-76 swaption pricing
- Implied vol inversion

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/bond/blob/main/notebooks/04_vol_surface.ipynb)

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np
import matplotlib.pyplot as plt
from bond import (
    sabr_vol, fit_sabr, vol_surface,
    black_swaption, black_swaption_vol,
    plot_smile, plot_surface,
)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('bond.vol loaded ✓')

## SABR Model Intuition

The SABR model (Hagan et al., 2002) is the industry standard for interest rate vol surfaces.

The four parameters control the smile shape:
- **α (alpha)**: overall vol level — higher α → higher ATM vol
- **β (beta)**: backbone shape (fixed at 0.5 by convention)
- **ρ (rho)**: smile skew — negative ρ → higher vol for lower strikes (typical in rates)
- **ν (nu)**: vol-of-vol — higher ν → more smile curvature

In [ ]:
F = 0.04   # ATM forward swap rate
T = 5.0    # 5Y expiry
beta = 0.5
strikes = np.linspace(0.01, 0.08, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Effect of rho
for rho, ls in [(-0.6, '-'), (0.0, '--'), (0.6, ':')]:
    vols = [sabr_vol(F, K, T, 0.20, beta, rho, 0.40) for K in strikes]
    axes[0].plot(strikes*100, np.array(vols)*100, ls, lw=2, label=f'ρ={rho}')
axes[0].axvline(F*100, color='gray', lw=0.8, ls='--')
axes[0].set_title('Effect of ρ (skew)')
axes[0].set_xlabel('Strike (%)'); axes[0].set_ylabel('Implied Vol (%)')
axes[0].legend()

# Effect of nu
for nu, ls in [(0.10, ':'), (0.40, '--'), (0.80, '-')]:
    vols = [sabr_vol(F, K, T, 0.20, beta, -0.30, nu) for K in strikes]
    axes[1].plot(strikes*100, np.array(vols)*100, ls, lw=2, label=f'ν={nu}')
axes[1].axvline(F*100, color='gray', lw=0.8, ls='--')
axes[1].set_title('Effect of ν (vol-of-vol / curvature)')
axes[1].set_xlabel('Strike (%)')
axes[1].legend()

# Effect of alpha
for alpha, ls in [(0.10, ':'), (0.20, '--'), (0.40, '-')]:
    vols = [sabr_vol(F, K, T, alpha, beta, -0.30, 0.40) for K in strikes]
    axes[2].plot(strikes*100, np.array(vols)*100, ls, lw=2, label=f'α={alpha}')
axes[2].axvline(F*100, color='gray', lw=0.8, ls='--')
axes[2].set_title('Effect of α (vol level)')
axes[2].set_xlabel('Strike (%)')
axes[2].legend()

plt.suptitle('SABR Parameter Sensitivity (F=4%, T=5Y, β=0.5)', y=1.01)
plt.tight_layout()
plt.show()

## Fitting SABR to Market Vols

In [ ]:
# Market vol quotes for 5Y expiry × 10Y tenor swaption
F = 0.044   # forward 10Y swap rate
T = 5.0
beta = 0.5

# Strikes expressed as spreads from ATM (in rate space)
market_vols = {
    0.020: 0.310,   # -200bp
    0.024: 0.285,   # -200bp ATM-200
    0.034: 0.248,   # -100bp
    0.044: 0.225,   # ATM
    0.054: 0.218,   # +100bp
    0.064: 0.220,   # +200bp
    0.074: 0.226,   # +300bp
}

# Calibrate
alpha, rho, nu = fit_sabr(market_vols, F=F, T=T, beta=beta)
print(f'Calibrated SABR parameters:')
print(f'  α (alpha) = {alpha:.4f}')
print(f'  β (beta)  = {beta:.4f}  [fixed]')
print(f'  ρ (rho)   = {rho:.4f}')
print(f'  ν (nu)    = {nu:.4f}')

print(f'\nFit quality:')
print(f'  {"Strike":>8}  {"Market":>8}  {"SABR":>8}  {"Error (bp)":>10}')
for K, v_mkt in market_vols.items():
    v_fit = sabr_vol(F, K, T, alpha, beta, rho, nu)
    print(f'  {K*100:>7.2f}%  {v_mkt*100:>7.2f}%  {v_fit*100:>7.2f}%  {(v_fit-v_mkt)*1e4:>+9.1f}bp')

In [ ]:
plot_smile(
    strikes=np.linspace(0.005, 0.10, 200),
    F=F, T=T, alpha=alpha, beta=beta, rho=rho, nu=nu,
    market_vols=market_vols,
    label=f'SABR (α={alpha:.3f}, ρ={rho:.3f}, ν={nu:.3f})',
)

## Building a Full Vol Surface

In [ ]:
# Calibrate SABR for multiple expiries
expiries = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
strikes = [0.020, 0.030, 0.035, 0.040, 0.045, 0.050, 0.055, 0.060, 0.070]

# Stylized market data: ATM vol term structure + smile
atm_vols = {1.0: 0.280, 2.0: 0.260, 3.0: 0.245, 5.0: 0.225, 7.0: 0.215, 10.0: 0.205}
forwards = {1.0: 0.043, 2.0: 0.044, 3.0: 0.044, 5.0: 0.044, 7.0: 0.045, 10.0: 0.045}

# Build market vol quotes: ATM vol + smile shape from SABR
sabr_params = {}
for T_exp in expiries:
    F_exp = forwards[T_exp]
    atm = atm_vols[T_exp]
    # Synthetic market: ATM vol + gentle smile
    mkt = {K: atm + 0.003 * ((K - F_exp) / F_exp) ** 2 * 100 + max(0, -0.002*(K - F_exp)/F_exp)
           for K in strikes}
    alpha_i, rho_i, nu_i = fit_sabr(mkt, F=F_exp, T=T_exp, beta=0.5)
    sabr_params[T_exp] = (alpha_i, rho_i, nu_i)
    print(f'{T_exp:4.0f}y  α={alpha_i:.3f}  ρ={rho_i:.3f}  ν={nu_i:.3f}')

# Build the surface matrix
surface = vol_surface(expiries, strikes, forwards, sabr_params, beta=0.5)
print(f'\nVol surface shape: {surface.shape}  (expiries × strikes)')

In [ ]:
plot_surface(expiries, strikes, surface, title='SABR Implied Vol Surface (swaptions)')

## Black-76 Swaption Pricing

In [ ]:
# Price a 5Y-into-5Y payer swaption
from bond import bootstrap, par_rate, annuity

deposits = {0.25: 0.053, 0.5: 0.0525}
swap_rates = {1:0.051, 2:0.049, 3:0.0475, 5:0.0455, 7:0.0445, 10:0.044}
curve = bootstrap(deposits, swap_rates)

F = par_rate(curve, 10.0)   # 10Y forward swap rate (started in 5Y)
A = annuity(curve, 10.0)    # 10Y annuity
K = 0.045                   # strike
T_exp = 5.0                 # 5Y expiry
notional = 10_000_000

# Get SABR vol at this strike
alpha_5, rho_5, nu_5 = sabr_params[5.0]
sigma = sabr_vol(F, K, T_exp, alpha_5, 0.5, rho_5, nu_5)
print(f'Forward swap rate:  {F:.4%}')
print(f'Strike:             {K:.4%}')
print(f'Annuity (10Y):      {A:.4f}')
print(f'SABR implied vol:   {sigma:.4%}')
print()

payer_pv = black_swaption(F, K, sigma, T_exp, A, notional, payer=True)
receiver_pv = black_swaption(F, K, sigma, T_exp, A, notional, payer=False)
print(f'Payer swaption PV:   ${payer_pv:>12,.0f}')
print(f'Receiver swaption PV:${receiver_pv:>12,.0f}')
print(f'Forward value check: ${(F-K)*A*notional:>12,.0f}  (should equal payer - receiver)')

# Verify put-call parity
parity_check = abs((payer_pv - receiver_pv) - (F-K)*A*notional)
print(f'Put-call parity error: ${parity_check:.4f}')

In [ ]:
# Swaption price and implied vol by strike
strikes_range = np.linspace(0.02, 0.07, 100)
payer_pvs = [black_swaption(F, K, sabr_vol(F, K, T_exp, alpha_5, 0.5, rho_5, nu_5),
                             T_exp, A, notional) for K in strikes_range]
implied_vols = [sabr_vol(F, K, T_exp, alpha_5, 0.5, rho_5, nu_5) * 100 for K in strikes_range]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(strikes_range*100, np.array(payer_pvs)/1e6, lw=2, color='#1565C0')
ax1.axvline(F*100, color='red', ls='--', lw=1, label=f'ATM={F:.3%}')
ax1.set_xlabel('Strike (%)'); ax1.set_ylabel('PV ($M)')
ax1.set_title('5Y×10Y Payer Swaption PV — $10M Notional')
ax1.legend()

ax2.plot(strikes_range*100, implied_vols, lw=2, color='#2E7D32')
ax2.axvline(F*100, color='red', ls='--', lw=1, label=f'ATM={F:.3%}')
ax2.set_xlabel('Strike (%)'); ax2.set_ylabel('Implied Vol (%)')
ax2.set_title('SABR Vol Smile — 5Y expiry')
ax2.legend()
plt.tight_layout()
plt.show()